# Rust hybrid engine: regime map

_Investigation `rust-hybrid-regime-map` — coder reproduction notebook._

**Question.** For what kinds of models does a pure-Rust hybrid ODE/SSA engine pay off, and
where does the hybrid partition stay accurate? The viva-hybrid trial measured a
single benchmark (Rust 2.01x vs numba -> MARGINAL); this investigation sweeps
model characteristics to map WHERE a Rust hybrid engine clears the >=5x bar AND
the hybrid approximation remains faithful to exact stochastic simulation.

A regime map over model characteristics. Each grid cell runs the same synthetic
hybrid model through the Rust engine and the Python baselines (N replicates),
recording Rust-vs-numba / Rust-vs-plain speedup and hybrid-vs-exact-SSA fidelity.
The maps shade speedup, draw the >=5x and <2x contours, and mask cells where the
hybrid is not faithful.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Language/engine is the lever (rewrite in Rust)**
- **Throughput/parallelism is the lever (not per-run speed)**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/viva-hybrid--copy-number-sweep').is_dir():
    REPO = Path('/Users/eranagmon/code/viva-hybrid--copy-number-sweep')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_hybrid.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Rust engine runs the hybrid model in the workbench (`rust-engine-in-workbench`)

**Question.** Does the real compiled Rust hybrid engine run a hybrid ODE/SSA model end-to-end as a
process-bigraph composite inside the vivarium-workbench, and how does its per-run speed
compare to plain Python?

**Objective.** Wrap the Rust engine as a pbg Step, expose it as a composite generator, run it through
the workbench; establish the integration and the plain-Python speedup baseline.

**Claim.** The Rust engine runs the bursting gene-expression benchmark as a workbench composite
(RustHybridStep bridging the real run_ir binary) and is ~30-50x faster than plain
interpreted Python per run.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `benchmark` | `viva_hybrid.composites.regime.rust_hybrid_benchmark` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_hybrid.composites.regime.rust_hybrid_benchmark`**

Rust hybrid ODE/SSA engine on the bursting gene-expression benchmark; emits per-species final means plus the engine's own simulate wall-time and RK4-step / stochastic-event counts.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_hybrid_composites_regime_rust_hybrid_benchmark = {
    'n_replicates': 50,   # Number of replicate seeds
    'dt'          : 0.1,   # Fixed RK4 step size (minutes)
    't_end'       : 300.0,   # Simulation horizon (minutes)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: rust-engine-in-workbench ===
STUDY = 'rust-engine-in-workbench'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_benchmark = 100
INTERVAL_benchmark = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_hybrid_composites_regime_rust_hybrid_benchmark if 'ov_viva_hybrid_composites_regime_rust_hybrid_benchmark' in dir() else None  # edits from the panel cell, if any
            spec_viva_hybrid_composites_regime_rust_hybrid_benchmark = materialize_composite('viva_hybrid.composites.regime.rust_hybrid_benchmark', _ov)
            _comp = Composite({'state': spec_viva_hybrid_composites_regime_rust_hybrid_benchmark['state']}, core=core)
            _comp.run(STEPS_benchmark)
        print('ran', 'benchmark', f'(viva_hybrid.composites.regime.rust_hybrid_benchmark) for', STEPS_benchmark, 'steps')
    except Exception as _e:
        print(f"benchmark not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Speedup landscape: event density x stochastic fraction (`speedup-landscape`)

**Question.** Across event density and stochastic fraction (system size fixed at 10), how large is
the Rust-vs-numba speedup, and where is the hybrid faithful to exact SSA?

**Claim.** Rust/numba stays 1.3-2.6x and is nearly flat in event density; the hybrid is faithful to
exact SSA only at stochastic_fraction = 1.0 (where it is the exact path).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `swept-models` | `viva_hybrid.composites.regime.rust_hybrid_benchmark` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_hybrid.composites.regime.rust_hybrid_benchmark`**

Rust hybrid ODE/SSA engine on the bursting gene-expression benchmark; emits per-species final means plus the engine's own simulate wall-time and RK4-step / stochastic-event counts.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_hybrid_composites_regime_rust_hybrid_benchmark = {
    'n_replicates': 50,   # Number of replicate seeds
    'dt'          : 0.1,   # Fixed RK4 step size (minutes)
    't_end'       : 300.0,   # Simulation horizon (minutes)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: speedup-landscape ===
STUDY = 'speedup-landscape'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_swept_models = 100
INTERVAL_swept_models = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_hybrid_composites_regime_rust_hybrid_benchmark if 'ov_viva_hybrid_composites_regime_rust_hybrid_benchmark' in dir() else None  # edits from the panel cell, if any
            spec_viva_hybrid_composites_regime_rust_hybrid_benchmark = materialize_composite('viva_hybrid.composites.regime.rust_hybrid_benchmark', _ov)
            _comp = Composite({'state': spec_viva_hybrid_composites_regime_rust_hybrid_benchmark['state']}, core=core)
            _comp.run(STEPS_swept_models)
        print('ran', 'swept-models', f'(viva_hybrid.composites.regime.rust_hybrid_benchmark) for', STEPS_swept_models, 'steps')
    except Exception as _e:
        print(f"swept-models not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**speedup-event-density-map**


In [ ]:
# speedup-event-density-map
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Regime frontier: system size x stochastic fraction (`regime-frontier`)

**Question.** Sweeping system size against stochastic fraction (density 100), is there any cell that is
both >=5x faster than numba AND faithful to exact SSA?

**Claim.** No. The only >=5x cells are large systems at low stochastic fraction (10.8x at size 100 /
fraction 0.1; 6.8x at fraction 0.25), and both FAIL fidelity (worst |z| ~14-17). The
faithful cells (stochastic_fraction = 1.0) are ~1.0x. Fast and trustworthy are disjoint.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `swept-models` | `viva_hybrid.composites.regime.rust_hybrid_benchmark` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_hybrid.composites.regime.rust_hybrid_benchmark`**

Rust hybrid ODE/SSA engine on the bursting gene-expression benchmark; emits per-species final means plus the engine's own simulate wall-time and RK4-step / stochastic-event counts.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_hybrid_composites_regime_rust_hybrid_benchmark = {
    'n_replicates': 50,   # Number of replicate seeds
    'dt'          : 0.1,   # Fixed RK4 step size (minutes)
    't_end'       : 300.0,   # Simulation horizon (minutes)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: regime-frontier ===
STUDY = 'regime-frontier'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_swept_models = 100
INTERVAL_swept_models = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_hybrid_composites_regime_rust_hybrid_benchmark if 'ov_viva_hybrid_composites_regime_rust_hybrid_benchmark' in dir() else None  # edits from the panel cell, if any
            spec_viva_hybrid_composites_regime_rust_hybrid_benchmark = materialize_composite('viva_hybrid.composites.regime.rust_hybrid_benchmark', _ov)
            _comp = Composite({'state': spec_viva_hybrid_composites_regime_rust_hybrid_benchmark['state']}, core=core)
            _comp.run(STEPS_swept_models)
        print('ran', 'swept-models', f'(viva_hybrid.composites.regime.rust_hybrid_benchmark) for', STEPS_swept_models, 'steps')
    except Exception as _e:
        print(f"swept-models not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**regime-frontier-map**


In [ ]:
# regime-frontier-map
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Copy-number frontier: does high copy open a faithful-and-fast window? (`copy-number-frontier`)

**Question.** The regime-frontier map found the only >=5x cells at large system size + low stochastic
fraction, all fidelity FAIL, and flagged copy number (held ~10-30) as the untested axis most
likely to open a faithful-and-fast window: high-copy continuous species should be well
approximated deterministically. Sweeping copy-number center N0 from 20 to 1000 at that
speedup corner (system_size=100, density=100), does raising copy number make any >=5x cell
faithful to exact SSA?

**Claim.** No. The map is FLAT in copy number. At N0 from 20 to 1000, every >=5x cell (stochastic
fraction <= 0.5; 10-11x at fraction 0.1, ~7x at 0.25) FAILS fidelity with worst |z| ~16, and
the value barely moves across a 50x copy-number range. The failure is entirely VARIANCE: the
deterministic mean is faithful at every copy number (worst |z_mean| ~2.5-5, at/below the |z|<=4
gate), but the hybrid gives continuous species ~zero intrinsic variance while exact SSA has
variance ~N0, so worst |z_var| stays ~16 regardless of copy number. The gap is copy-number-
invariant by construction, so copy number was never going to close it. Faithful cells remain
only at stochastic_fraction = 1.0 (fully stochastic, ~1x). 0 of 16 cells are both >=5x and
faithful. This resolves the open decision from regime-frontier and strengthens the
investigation verdict: no faithful-and-fast regime exists even with copy number swept.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `swept-models` | `viva_hybrid.composites.regime.rust_hybrid_benchmark` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_hybrid.composites.regime.rust_hybrid_benchmark`**

Rust hybrid ODE/SSA engine on the bursting gene-expression benchmark; emits per-species final means plus the engine's own simulate wall-time and RK4-step / stochastic-event counts.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_hybrid_composites_regime_rust_hybrid_benchmark = {
    'n_replicates': 50,   # Number of replicate seeds
    'dt'          : 0.1,   # Fixed RK4 step size (minutes)
    't_end'       : 300.0,   # Simulation horizon (minutes)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: copy-number-frontier ===
STUDY = 'copy-number-frontier'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_swept_models = 100
INTERVAL_swept_models = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_hybrid_composites_regime_rust_hybrid_benchmark if 'ov_viva_hybrid_composites_regime_rust_hybrid_benchmark' in dir() else None  # edits from the panel cell, if any
            spec_viva_hybrid_composites_regime_rust_hybrid_benchmark = materialize_composite('viva_hybrid.composites.regime.rust_hybrid_benchmark', _ov)
            _comp = Composite({'state': spec_viva_hybrid_composites_regime_rust_hybrid_benchmark['state']}, core=core)
            _comp.run(STEPS_swept_models)
        print('ran', 'swept-models', f'(viva_hybrid.composites.regime.rust_hybrid_benchmark) for', STEPS_swept_models, 'steps')
    except Exception as _e:
        print(f"swept-models not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**copy-number-frontier-map**


In [ ]:
# copy-number-frontier-map
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Open decisions
- Adopt a variance-tolerant fidelity criterion (mean-only / relative-variance), or redesign the partition?
